# Kalshi Coverage


In [42]:
import requests
import pandas as pd
import numpy as np
import time
import yfinance as yf

In [43]:

BASE = "https://api.elections.kalshi.com/trade-api/v2"
ET = "America/New_York"
MAX_SPREAD = 0.20

def get_json(url, params=None):
    # Make a GET request to the specified URL with optional parameters and return the JSON response.
    # returns None for 404; a market missing
    for attempt in range(6): # retry up to 6 times
        response = requests.get(url, params=params, timeout=60)
        if response.status_code == 429: # too many requests: wait and retry
            time.sleep(2 ** attempt) # wait for - second before retrying
            continue        
        if response.status_code == 404:
            return None
        response.raise_for_status() # Raise an exception for other HTTP errors
        return response.json()
    raise RuntimeError(f"Failed to get JSON from {url} after 6 attempts.")

events = get_json(f"{BASE}/events", params={"series_ticker": "KXFED", "limit": 200})["events"]
meetings = []

print(events[0])
print("Number of meetings:", len(events))

missing = 0
for e in events:
    if e.get("strike_date") is None:
        missing += 1
    print(e.get("event_ticker"), e.get("strike_date"))

print("Missing meetings:", missing)

{'category': 'Economics', 'collateral_return_type': 'DIRECNET', 'event_ticker': 'KXFED-28JAN', 'exchange_index': 0, 'last_updated_ts': '2026-10-27T19:53:03.069314Z', 'mutually_exclusive': False, 'product_metadata': {'competition': 'US FED', 'competition_scope': 'Fed Rate After Meeting'}, 'series_ticker': 'KXFED', 'settlement_sources': [{'name': 'Federal Reserve Board of Governors', 'url': 'https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm'}], 'strike_date': '2028-01-26T19:00:00Z', 'strike_period': '', 'sub_title': 'On Jan 26, 2028', 'title': 'Fed funds rate after Jan 2028 meeting?'}
Number of meetings: 52
KXFED-28JAN 2028-01-26T19:00:00Z
KXFED-27DEC 2027-12-08T19:00:00Z
KXFED-27OCT 2027-10-27T18:00:00Z
KXFED-27SEP 2027-09-15T18:00:00Z
KXFED-27JUL 2027-07-28T18:00:00Z
KXFED-27JUN 2027-06-09T18:00:00Z
KXFED-27APR 2027-04-28T18:00:00Z
KXFED-27MAR 2027-03-17T18:00:00Z
KXFED-27JAN 2027-01-27T19:00:00Z
KXFED-26DEC 2026-12-09T19:00:00Z
KXFED-26OCT 2026-10-28T18:00:00Z
KXFED-26SE

In [44]:

markets = get_json(f"{BASE}/historical/markets", params={"event_ticker": "FED-22MAR", "limit": 200})["markets"]
print("Number of markets:", len(markets))
print(markets[0]["ticker"], markets[0]["close_time"])

Number of markets: 3
FED-22MAR-T0.75 2022-03-16T17:55:00Z


In [45]:
for m in markets:
    print(m.get("ticker"), m.get("close_time"))

def strike_of(ticker):
    # Extract the strike price from the market ticker.
    # Example: "FED-22MAR-T0.25" -> 0.25
    return float(ticker.split("-T")[-1])

print(strike_of("FED-22MAR-T0.75"))    # 0.75
print(strike_of("FED-22MAR-T1"))       # 1.0
print(strike_of("KXFED-26SEP-T3.75"))  

FED-22MAR-T0.75 2022-03-16T17:55:00Z
FED-22MAR-T0.5 2022-03-16T17:55:00Z
FED-22MAR-T0.25 2022-03-16T17:55:00Z
0.75
1.0
3.75


In [46]:
def get_markets(event_ticker):
    # Fetch all markets for a given event ticker
    markets = (get_json(f"{BASE}/historical/markets", params={"event_ticker": event_ticker, "limit": 200}) or {}).get("markets", [])    
    if not markets:
        markets = (get_json(f"{BASE}/markets", params={"event_ticker": event_ticker, "limit": 200}) or {}).get("markets", [])
    
    return markets

print(len(get_markets("FED-22MAR")))     # 3
print(len(get_markets("KXFED-26DEC")))   # about 11

3
11


In [47]:
for e in events:
    markets = get_markets(e.get("event_ticker"))
    if not markets:
        continue

    meeting_time = pd.to_datetime(markets[0]["close_time"], utc=True).tz_convert(ET)

    strikes = {}
    for m in markets:
        strikes[strike_of(m["ticker"])] = m["ticker"]

    meetings.append({
        "event_ticker": e.get("event_ticker"),
        "time": meeting_time,
        "markets": strikes
    })
    time.sleep(0.1)  # Sleep for 100ms to avoid hitting rate limits

meetings.sort(key=lambda m: m["time"])
print(len(meetings), "meetings found")
print(meetings[0]["event"], meetings[0]["time"], meetings[0]["markets"])
print(meetings[-1]["event"], meetings[-1]["time"])

52 meetings found


KeyError: 'event'